## Section 1: Setup and Data Structures
Run this first. It defines imports, paths, and dataclasses used later.


In [ ]:
from __future__ import annotations

import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np



if Path("data").exists():
    DATA_DIR = Path("data")
elif Path("../data").exists():
    DATA_DIR = Path("../data")
else:
    DATA_DIR = Path.cwd() / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = Path("submission.csv")


@dataclass
class Standardizer:
    mean_: np.ndarray
    std_: np.ndarray


@dataclass
class PCAState:
    components_: np.ndarray  # shape: (k, d)
    explained_variance_: np.ndarray  # shape: (k,)


@dataclass
class LinearModel:
    w: np.ndarray
    b: float



## Section 2: Data Loading and Basic Preprocessing
Run this cell. It includes CSV loading, train/validation split, and standardization helpers.


In [2]:
def load_train(path: Path) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Returns (ids, X, y)."""
    ids, X_rows, y = [], [], []
    with path.open() as f:
        reader = csv.DictReader(f)
        feature_cols = [c for c in reader.fieldnames if c.startswith("f_")]
        for row in reader:
            ids.append(int(row["id"]))
            X_rows.append([float(row[c]) for c in feature_cols])
            y.append(float(row["target"]))
    return np.array(ids), np.array(X_rows, dtype=float), np.array(y, dtype=float)


def load_test(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (ids, X)."""
    ids, X_rows = [], []
    with path.open() as f:
        reader = csv.DictReader(f)
        feature_cols = [c for c in reader.fieldnames if c.startswith("f_")]
        for row in reader:
            ids.append(int(row["id"]))
            X_rows.append([float(row[c]) for c in feature_cols])
    return np.array(ids), np.array(X_rows, dtype=float)


def train_val_split(
    X: np.ndarray,
    y: np.ndarray,
    val_ratio: float = 0.2,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = X.shape[0]
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    n_val = int(n * val_ratio)
    val_idx, train_idx = perm[:n_val], perm[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


def fit_standardizer(X: np.ndarray, eps: float = 1e-8) -> Standardizer:
    mean_ = X.mean(axis=0)
    std_ = X.std(axis=0)
    std_ = np.where(std_ < eps, 1.0, std_)
    return Standardizer(mean_, std_)


def apply_standardizer(X: np.ndarray, scaler: Standardizer) -> np.ndarray:
    return (X - scaler.mean_) / scaler.std_




## Section 3: PCA Implementation
Implement `fit_pca` and `transform_pca`.

Guidance:
- Center features.
- Compute covariance matrix.
- Use eigen-decomposition.
- Sort components by descending eigenvalue.
- Keep top `n_components`.


In [ ]:
def fit_pca(X: np.ndarray, n_components: int) -> PCAState:

    # 1) Center X
    mean = np.mean(X, axis=0)
    X_centered = X - mean

    # 2) Covariance matrix
    n_samples = X_centered.shape[0]
    covariance_matrix = (X_centered.T @ X_centered) / (n_samples - 1)

    # 3) Eigen-decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

    # 4) Sort eigenvalues descending
    sorted_indices = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_indices]
    eigenvectors = eigenvectors[:, sorted_indices]

    # 5) Keep top n_components
    components_ = eigenvectors[:, :n_components].T  # transpose to match (k, d)
    explained_variance_ = eigenvalues[:n_components]

    # Return PCAState with components and explained variance
    return PCAState(components_, explained_variance_)


def transform_pca(X: np.ndarray, pca: PCAState) -> np.ndarray:
    
    # If components_ has shape (k, d), output should be shape (n, k).
    # Center using training mean
    return X @ pca.components_.T
    #raise NotImplementedError("Implement PCA transform here.")




## Section 4: Prediction, Metrics, and Gradients
Run this as-is. It defines model prediction, RMSE, and gradients of linear model parameters for MSE + L2 objective function.


In [4]:
def predict(X: np.ndarray, model: LinearModel) -> np.ndarray:
    return X @ model.w + model.b


def mse_loss(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((y_true - y_pred) ** 2))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mse_loss(y_true, y_pred)))



def loss_and_gradients(
    X: np.ndarray,
    y: np.ndarray,
    model: LinearModel,
    l2: float = 0.0,
) -> Tuple[float, np.ndarray, float]:
    """Returns (loss, dw, db) for MSE + L2."""
    n = X.shape[0]
    y_hat = predict(X, model)
    residual = y_hat - y
    loss = float(np.mean(residual**2) + 0.5 * l2 * np.sum(model.w * model.w))

    dw = (2.0 / n) * (X.T @ residual) + l2 * model.w
    db = float((2.0 / n) * np.sum(residual))
    return loss, dw, db


def init_model(d: int, seed: int = 42) -> LinearModel:
    rng = np.random.default_rng(seed)
    w = rng.normal(0.0, 0.01, size=d)
    b = 0.0
    return LinearModel(w=w, b=b)




## Section 5: Batch Gradient Descent
Implement one full-batch update per epoch and track loss history.


In [5]:
def fit_gradient_descent(
    X: np.ndarray,
    y: np.ndarray,
    lr: float = 0.03,
    epochs: int = 400,
    l2: float = 1e-4,
    seed: int = 42,
) -> Tuple[LinearModel, Dict[str, list]]:
    """Full-batch Gradient Descent for linear regression."""

    # Set random seed for reproducibility
    rng = np.random.default_rng(seed)

    n, d = X.shape  # n = number of samples, d = number of features

    # Initialise weights randomly and bias to 0
    w = rng.standard_normal(d)
    b = 0.0

    # Store training loss over time
    history = {"train_loss": []}

    # Loop over epochs (full passes through dataset)
    for _ in range(epochs):

        # Compute predictions for ALL data
        y_pred = X @ w + b

        # Compute error (difference between prediction and true value)
        error = y_pred - y

        # Compute loss (MSE + L2 regularisation)
        loss = (error**2).mean() + (l2 / 2) * (w @ w)
        history["train_loss"].append(loss)

        # Compute gradients (how to change w and b)
        grad_w = (2 / n) * (X.T @ error) + l2 * w
        grad_b = (2 / n) * error.sum()

        # Update parameters (move downhill)
        w -= lr * grad_w
        b -= lr * grad_b

    # Return trained model and loss history
    return LinearModel(w=w, b=b), history

## Section 6: SGD
Implement mini-batch updates in `fit_sgd`.


In [6]:
def iterate_minibatches(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    rng: np.random.Generator,
):
    n = X.shape[0]
    idx = rng.permutation(n)
    for start in range(0, n, batch_size):
        batch_idx = idx[start : start + batch_size]
        yield X[batch_idx], y[batch_idx]

def fit_sgd(
    X: np.ndarray,
    y: np.ndarray,
    lr: float = 0.01,
    epochs: int = 120,
    batch_size: int = 64,
    l2: float = 1e-4,
    seed: int = 42,
) -> Tuple[LinearModel, Dict[str, list]]:
    """Mini-batch Stochastic Gradient Descent."""

    # Random generator for shuffling data
    rng = np.random.default_rng(seed)

    n, d = X.shape

    # Initialise weights and bias
    w = rng.standard_normal(d)
    b = 0.0

    history = {"train_loss": []}

    # Loop over epochs
    for _ in range(epochs):

        # Loop over mini-batches
        for X_batch, y_batch in iterate_minibatches(X, y, batch_size, rng):

            m = X_batch.shape[0]  # batch size

            # Predictions for this mini-batch only
            y_pred = X_batch @ w + b

            # Error for this batch
            error = y_pred - y_batch

            # Compute gradients using batch
            grad_w = (2 / m) * (X_batch.T @ error) + l2 * w
            grad_b = (2 / m) * error.sum()

            # Update weights immediately (many updates per epoch)
            w -= lr * grad_w
            b -= lr * grad_b

        # After full epoch, compute total loss on ALL data (for tracking)
        y_full = X @ w + b
        full_error = y_full - y
        loss = (full_error**2).mean() + (l2 / 2) * (w @ w)
        history["train_loss"].append(loss)

    return LinearModel(w=w, b=b), history

## Section 7: Adam
Implement first/second moments, bias correction, and Adam parameter updates.


In [ ]:
def fit_adam(
    X: np.ndarray,
    y: np.ndarray,
    lr: float = 0.01,
    epochs: int = 120,
    batch_size: int = 64,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
    l2: float = 1e-4,
    seed: int = 42,
) -> Tuple[LinearModel, Dict[str, list]]:
    
    rng = np.random.default_rng(seed)
    n, d = X.shape

   # Initialise parameters using init_model for consistency
    model_init = init_model(d, seed=seed)
    w = model_init.w.copy()
    b = model_init.b

    # Adam moment estimates
    m_w = np.zeros(d)
    v_w = np.zeros(d)
    m_b = 0.0
    v_b = 0.0

    history = {"train_loss": []}  # fixed key name
    t = 0  # global timestep

    for _ in range(epochs):
        for X_batch, y_batch in iterate_minibatches(X, y, batch_size, rng):
            m = X_batch.shape[0]
            t += 1

            # Forward pass
            y_pred = X_batch @ w + b
            error = y_pred - y_batch

            # Gradients (MSE + L2)
            grad_w = (2 / m) * (X_batch.T @ error) + l2 * w
            grad_b = (2 / m) * np.sum(error)

            # Update first moment (mean)
            m_w = beta1 * m_w + (1 - beta1) * grad_w
            m_b = beta1 * m_b + (1 - beta1) * grad_b

            # Update second moment (uncentred variance)
            v_w = beta2 * v_w + (1 - beta2) * (grad_w ** 2)
            v_b = beta2 * v_b + (1 - beta2) * (grad_b ** 2)

            # Bias correction
            m_w_hat = m_w / (1 - beta1 ** t)
            m_b_hat = m_b / (1 - beta1 ** t)
            v_w_hat = v_w / (1 - beta2 ** t)
            v_b_hat = v_b / (1 - beta2 ** t)

            # Parameter updates
            w -= lr * m_w_hat / (np.sqrt(v_w_hat) + eps)
            b -= lr * m_b_hat / (np.sqrt(v_b_hat) + eps)

        # Full-dataset loss at end of each epoch
        y_full = X @ w + b
        full_error = y_full - y
        loss = float(np.mean(full_error ** 2) + 0.5 * l2 * np.sum(w ** 2))
        history["train_loss"].append(loss)

    return LinearModel(w=w, b=b), history

## Section 8: Training and Submission Pipeline


In [ ]:
def write_submission(ids: np.ndarray, preds: np.ndarray, out_path: Path) -> None:
    with out_path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "prediction"])
        for i, p in zip(ids, preds):
            writer.writerow([int(i), float(p)])


#
SEED = 266

VAL_RATIO = 0.2
# Tune number of principal components N_COMPONENTS here
N_COMPONENTS = 60

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Could not find data files at {TRAIN_PATH} and {TEST_PATH}. "
        "Run notebook from project root or starter/"
    )

_, X, y = load_train(TRAIN_PATH)
test_ids, X_test = load_test(TEST_PATH)

X_tr, y_tr, X_val, y_val = train_val_split(X, y, val_ratio=VAL_RATIO, seed=SEED)

scaler = fit_standardizer(X_tr)
X_tr_s = apply_standardizer(X_tr, scaler)
X_val_s = apply_standardizer(X_val, scaler)
X_test_s = apply_standardizer(X_test, scaler)

# fit PCA on X_tr_s only and transform all splits.
pca = fit_pca(X_tr_s, n_components=N_COMPONENTS)
X_tr_p = transform_pca(X_tr_s, pca)
X_val_p = transform_pca(X_val_s, pca)
X_test_p = transform_pca(X_test_s, pca)

# train all three optimizers and compare validation RMSE.
model_gd, hist_gd = fit_gradient_descent(X_tr_p, y_tr)
model_sgd, hist_sgd = fit_sgd(X_tr_p, y_tr)
model_adam, hist_adam = fit_adam(X_tr_p, y_tr)

# select best model based on validation RMSE.
val_preds_gd = predict(X_val_p, model_gd)
val_rmse_gd = rmse(y_val, val_preds_gd)

val_preds_sgd = predict(X_val_p, model_sgd)
val_rmse_sgd = rmse(y_val, val_preds_sgd)

val_preds_adam = predict(X_val_p, model_adam)
val_rmse_adam = rmse(y_val, val_preds_adam)

print(f"GD RMSE:   {val_rmse_gd:.6f}")
print(f"SGD RMSE:  {val_rmse_sgd:.6f}")
print(f"Adam RMSE: {val_rmse_adam:.6f}")

# Select best model
best_rmse = min(val_rmse_gd, val_rmse_sgd, val_rmse_adam)

if best_rmse == val_rmse_gd:
    model = model_gd
    print("Selected: Gradient Descent")
elif best_rmse == val_rmse_sgd:
    model = model_sgd
    print("Selected: SGD")
else:
    model = model_adam
    print("Selected: Adam")

# select best model based on validation RMSE.
# Replace this placeholder after implementation.
#model = init_model(X_tr_p.shape[1])

val_preds = predict(X_val_p, model)
val_rmse = rmse(y_val, val_preds)
print(f"Best Validation RMSE: {best_rmse:.6f}")

test_preds = predict(X_test_p, model)
write_submission(test_ids, test_preds, SUBMISSION_PATH)
print(f"Wrote: {SUBMISSION_PATH.resolve()}")

FileNotFoundError: Could not find data files at c:\Users\sohil\Downloads\cw2\cs266\data\train.csv and c:\Users\sohil\Downloads\cw2\cs266\data\test.csv. Run notebook from project root or starter/